# Tarea 2 – Parte 1: Corpus y extracción (OCR + ASR)

**Pregunta guía:** ¿pueden convertirse documentos escaneados y entrevistas en audio en un archivo de texto estructurado y buscable?

| Configuración | Valor |
| :--- | :--- |
| Fecha de extracción | **2026-10-23** (hora de Lima) |
| Corpus A – documentos | 25 imágenes de documentos peruanos: 8 impresos nítidos, 8 fotos de celular y 9 manuscritos |
| Corpus B – audio | 23 clips (48,9 min) en español: 2 audios de YouTube, 9 entrevistas propias y 12 frases de OpenSLR 73 |
| OCR | **PaddleOCR 3.7.0** (PaddleX 3.7.2): detector `PP-OCRv6_medium_det` + reconocedor `PP-OCRv6_medium_rec`, `lang="es"`, motor PyTorch (`engine="transformers"`) en CPU |
| ASR | **faster-whisper 1.2.1** (CTranslate2 4.6.0) con **Whisper large-v3-turbo** en int8, CPU, `beam_size=5`, `language="es"` |
| VAD | Silero VAD de faster-whisper (`vad_filter=True`, pausas de 300 ms) |
| Diarización | Huellas de voz WeSpeaker ResNet34 (`pyannote.audio` 4.0.7) + clustering jerárquico por coseno, adaptado de `voz.py` (Sesión 13) |
| Seudonimización | Candidatos con spaCy `es_core_news_md` 3.8.0 (NER) y expresiones regulares, revisión manual y reemplazo difuso con `seudonimizar` de `voz.py` |

**Fuentes y licencias** (detalle por archivo en `datos/fuentes.csv`):

| Archivos | Fuente | Licencia |
| :--- | :--- | :--- |
| `impreso1`–`impreso8` | Págs. 2, 4, 9, 15, 16, 22, 32 y 36 de las bases integradas de la AS N° 013-2024-INDECOPI-1, buscador público del SEACE | Texto oficial sin derecho de autor (D. Leg. 822, art. 9 b) |
| `foto1`–`foto8` | Fotos con celular, tomadas por la estudiante, de la Directiva Ejecutiva Regional N° 002-2024-GR/GOB.REG.TACNA | Texto oficial sin derecho de autor (D. Leg. 822, art. 9 b) |
| `manuscrito3`, `6`, `8`, `9` | Wikimedia Commons (Academia Nacional de la Historia de Argentina y Biblioteca Nacional del Perú) | Dominio público o CC0 |
| `manuscrito1`, `2`, `4`, `5`, `7` | Notas de la BNP, El Peruano, BYU y Revista de Arte | Texto en dominio público; fotografía con derechos reservados: **uso académico local, sin redistribución** |
| 2 audios `.mp4` | Grabación propia de videos de YouTube, según la estudiante | Licencia del video por confirmar: **uso académico local, sin redistribución** |
| 9 audios `.ogg` / `.aac` | Entrevistas breves grabadas por la estudiante | Grabación propia con consentimiento informado, según la estudiante; el audio no se publica |
| 12 clips `.wav` | OpenSLR 73, *Crowdsourced high-quality Peruvian Spanish speech data set* (Google) | CC BY-SA 4.0; Guevara-Rukoz et al. (2020), LREC |

**Entorno.** Las versiones exactas están en `requirements-notebooks.txt`. En esta laptop (Windows 11, solo CPU, 7,3 GB de RAM) se fijaron `torch==2.8.0`, `ctranslate2==4.6.0` y `onnxruntime==1.20.1`, porque las versiones más nuevas no cargan con el runtime de Visual C++ instalado (`WinError 1114`). Por el mismo motivo PaddleOCR corre con el motor PyTorch: con PaddlePaddle 3.3.1 y oneDNN el detector falla en CPU.

**Privacidad.** Este notebook se publica en GitHub con sus salidas. Por eso solo muestra textos ya seudonimizados y conteos. La tabla de nombres reales y sus seudónimos, los textos crudos y los audios se quedan en `datos/raw/`, que no se versiona.

In [1]:
import gc
import hashlib
import io
import json
import os
import random
import re
import time
import urllib.request
import urllib.robotparser
import warnings
import zipfile
from datetime import date
from importlib.metadata import version
from pathlib import Path

# Antes de importar los modelos: sin chequeo de servidores de PaddleX ni barras de progreso
os.environ["PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK"] = "True"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.io as pio
import truststore
from PIL import Image

# Valida HTTPS con los certificados del sistema operativo. En Windows, el antivirus
# intercepta HTTPS con su propio certificado raíz, que el certifi de requests no conoce.
truststore.inject_into_ssl()

# Plotly: interactivo en Jupyter/VS Code + copia PNG para que GitHub muestre los gráficos
pio.renderers.default = "plotly_mimetype+png"
pio.renderers["png"].width = 950
pio.templates.default = "plotly_white"
pd.set_option("display.max_colwidth", 110)

USER_AGENT = ("Mozilla/5.0 (compatible; TareaDiplomadoPython/1.0; uso academico; "
              "+https://github.com/IvettexMamani/diplomado-python-2026)")

DATOS = Path("datos")
RAW = DATOS / "raw"
DOCS_RAW = RAW / "documentos"
AUDIO_RAW = RAW / "audio"
FECHA = date.today().isoformat()

PAQUETES = ["paddleocr", "paddlex", "torch", "faster-whisper", "ctranslate2", "onnxruntime",
            "pyannote.audio", "spacy", "es_core_news_md", "rapidfuzz", "plotly", "pandas"]
print("Fecha de ejecución:", FECHA)
pd.Series({p: version(p) for p in PAQUETES}, name="versión").to_frame()

Fecha de ejecución: 2026-10-23


,versión
paddleocr,3.7.0
paddlex,3.7.2
torch,2.8.0
faster-whisper,1.2.1
ctranslate2,4.6.0
onnxruntime,1.20.1
pyannote.audio,4.0.7
spacy,3.8.16
es_core_news_md,3.8.0
rapidfuzz,3.14.6


## 1. Corpus

### 1.1 Corpus A: 25 documentos

El corpus de documentos se armó el 2026-10-22. El tipo de cada documento sale del nombre de su archivo (`impresoN`, `fotoN`, `manuscritoN`).

In [2]:
TIPOS = ["impreso", "foto", "manuscrito"]
CUOTA = {"impreso": 8, "foto": 8, "manuscrito": 9}

docs = pd.DataFrame({"ruta": sorted(DOCS_RAW.glob("*.jpg"))})
docs["id"] = docs["ruta"].map(lambda r: r.stem)
docs["tipo"] = docs["id"].str.extract(r"^([a-z]+)", expand=False)
docs["n"] = docs["id"].str.extract(r"(\d+)$", expand=False).astype(int)
docs = docs.sort_values(["tipo", "n"], key=lambda s: s.map(TIPOS.index) if s.name == "tipo" else s)
docs = docs.reset_index(drop=True)
docs["ancho"], docs["alto"] = zip(*(Image.open(r).size for r in docs["ruta"]))

resumen_docs = docs.groupby("tipo", sort=False).agg(documentos=("id", "size"),
                                                     ancho_min=("ancho", "min"), ancho_max=("ancho", "max"))
resumen_docs["cuota"] = resumen_docs.index.map(CUOTA)
resumen_docs["cumple"] = resumen_docs["documentos"] >= resumen_docs["cuota"]
print(f"{len(docs)} documentos; los tres tipos cumplen la cuota: {resumen_docs['cumple'].all()}")
resumen_docs

25 documentos; los tres tipos cumplen la cuota: True


,documentos,ancho_min,ancho_max,cuota,cumple
tipo,,,,,
impreso,8,1241,1241,8,True
foto,8,960,960,8,True
manuscrito,9,524,4065,9,True


**Descartes durante la recolección (2026-10-22).** Se reemplazaron hasta completar la cuota y están registrados en los mensajes de los commits `5ca4ac6` y `54008e7`:

| Descartado | Tipo buscado | Motivo |
| :--- | :--- | :--- |
| Resolución 502-2026 | impreso | Datos personales |
| Págs. 1, 14 y 21 de las bases integradas | impreso | Datos personales (firmas y correo con nombres) |
| Un logotipo y un cartel | — | No son documentos |
| Dos cartas en una sola imagen (El Peruano) | manuscrito | Mezcla dos documentos |
| Página en inglés con marca de agua de 123RF | manuscrito | Licencia e idioma |
| Dos miniaturas de menos de 500 px | manuscrito | Resolución insuficiente |

### 1.2 Corpus B: descarga de OpenSLR 73

Los 12 clips de OpenSLR 73 completan los 20 clips mínimos y suman una condición de grabación distinta: voz leída y grabada con buena calidad, frente a las entrevistas por WhatsApp.

- **Buenas prácticas:** se lee `robots.txt` antes de descargar y se respeta su `Crawl-delay` de 5 s, que es mayor que la pausa de 1–2 s del enunciado. Todas las solicitudes usan el `User-Agent` del proyecto.
- **Descarga parcial:** cada zip pesa unos 900 MB. Con peticiones HTTP Range solo se piden los bytes de los 12 WAV elegidos. La clase `ZipRemoto` está adaptada de `datos.py` de la Sesión 13 (licencia MIT).
- **Selección:** 6 hablantes mujeres y 6 hombres, elegidos al azar con semilla 42. De cada hablante se toma su frase más larga que no se haya elegido antes, porque OpenSLR repite las mismas frases entre hablantes.

In [3]:
OPENSLR = "https://www.openslr.org"
DIR_OPENSLR = AUDIO_RAW / "openslr73"
DIR_OPENSLR.mkdir(parents=True, exist_ok=True)
REGISTRO_DESCARGA = DIR_OPENSLR / "descarga.json"

PAUSA_SEG = 1.5            # se sube al Crawl-delay de robots.txt apenas se lee
_ultima, _solicitudes = [0.0], [0]


def pedir(url, rango=None, metodo="GET"):
    "Solicitud respetuosa: User-Agent del proyecto y pausa mínima entre solicitudes."
    espera = PAUSA_SEG - (time.time() - _ultima[0])
    if espera > 0:
        time.sleep(espera)
    cabeceras = {"User-Agent": USER_AGENT}
    if rango:
        cabeceras["Range"] = f"bytes={rango[0]}-{rango[1]}"
    try:
        with urllib.request.urlopen(urllib.request.Request(url, headers=cabeceras, method=metodo), timeout=120) as r:
            return r.read() if metodo == "GET" else r.headers
    finally:
        _ultima[0] = time.time()
        _solicitudes[0] += 1


class ZipRemoto(io.RawIOBase):
    "Archivo remoto que se lee por partes (HTTP Range): zipfile solo pide los bytes que necesita."

    BLOQUE = 1 << 20  # zipfile hace muchas lecturas chicas: se piden bloques de 1 MB

    def __init__(self, url):
        self.url, self.pos, self.buf_ini, self.buf = url, 0, 0, b""
        self.size = int(pedir(url, metodo="HEAD")["Content-Length"])

    def readable(self):
        return True

    def seekable(self):
        return True

    def tell(self):
        return self.pos

    def seek(self, offset, whence=0):
        self.pos = {0: offset, 1: self.pos + offset, 2: self.size + offset}[whence]
        return self.pos

    def read(self, n=-1):
        if n is None or n < 0:
            n = self.size - self.pos
        n = min(n, self.size - self.pos)
        if n <= 0:
            return b""
        fin = self.pos + n
        if not (self.buf_ini <= self.pos and fin <= self.buf_ini + len(self.buf)):
            self.buf_ini = self.pos
            self.buf = pedir(self.url, (self.pos, min(self.pos + max(n, self.BLOQUE), self.size) - 1))
        datos = self.buf[self.pos - self.buf_ini:fin - self.buf_ini]
        self.pos += len(datos)
        return datos


def elegir_frases(tsv, n, rng, usados):
    "n hablantes al azar y, de cada uno, su frase más larga que no se haya elegido antes."
    por_hablante = {}
    for linea in tsv.decode("utf-8").strip().splitlines():
        fid, texto = linea.split("\t")[:2]
        por_hablante.setdefault(fid.split("_")[1], []).append((fid, texto.strip()))
    elegidas = []
    for h in rng.sample(sorted(por_hablante), n):
        libres = [x for x in por_hablante[h] if x[1] not in usados]
        fid, texto = max(libres, key=lambda x: len(x[1].split()))
        usados.add(texto)
        elegidas.append((fid, texto))
    return elegidas


indice_openslr = DIR_OPENSLR / "openslr73_transcripciones.tsv"
if not indice_openslr.exists():
    t0 = time.time()
    texto_robots = pedir(OPENSLR + "/robots.txt").decode("utf-8")
    (DIR_OPENSLR / "robots.txt").write_text(texto_robots, encoding="utf-8")
    robots = urllib.robotparser.RobotFileParser()
    robots.parse(texto_robots.splitlines())
    PAUSA_SEG = max(robots.crawl_delay(USER_AGENT) or 0, PAUSA_SEG)

    rng, usados, filas = random.Random(42), set(), []
    for genero in ["female", "male"]:
        url_tsv = f"{OPENSLR}/resources/73/line_index_{genero}.tsv"
        url_zip = f"{OPENSLR}/resources/73/es_pe_{genero}.zip"
        assert robots.can_fetch(USER_AGENT, url_tsv) and robots.can_fetch(USER_AGENT, url_zip)
        tsv = pedir(url_tsv)
        (DIR_OPENSLR / f"line_index_{genero}.tsv").write_bytes(tsv)
        zf = zipfile.ZipFile(ZipRemoto(url_zip))
        nombres = {Path(i.filename).stem: i for i in zf.infolist() if i.filename.endswith(".wav")}
        for fid, texto in elegir_frases(tsv, 6, rng, usados):
            (DIR_OPENSLR / f"{fid}.wav").write_bytes(zf.read(nombres[fid]))
            filas.append({"file_id": fid, "genero": genero, "zip_origen": Path(url_zip).name, "transcripcion": texto})
    pd.DataFrame(filas).to_csv(indice_openslr, sep="\t", index=False)
    REGISTRO_DESCARGA.write_text(json.dumps({
        "fecha": FECHA, "user_agent": USER_AGENT, "crawl_delay_robots": robots.crawl_delay(USER_AGENT),
        "pausa_seg": PAUSA_SEG, "solicitudes": _solicitudes[0], "minutos": round((time.time() - t0) / 60, 1),
    }, ensure_ascii=False, indent=1), encoding="utf-8")

openslr = pd.read_csv(indice_openslr, sep="\t")
registro = json.loads(REGISTRO_DESCARGA.read_text(encoding="utf-8"))
print(f"Descarga del {registro['fecha']}: {registro['solicitudes']} solicitudes a openslr.org en "
      f"{registro['minutos']} min, con pausa de {registro['pausa_seg']} s "
      f"(Crawl-delay de robots.txt: {registro['crawl_delay_robots']} s)")
print("User-Agent:", registro["user_agent"])
openslr[["file_id", "genero", "zip_origen"]].assign(palabras=openslr["transcripcion"].str.split().str.len())

Descarga del 2026-10-23: 23 solicitudes a openslr.org en 2.4 min, con pausa de 5 s (Crawl-delay de robots.txt: 5 s)
User-Agent: Mozilla/5.0 (compatible; TareaDiplomadoPython/1.0; uso academico; +https://github.com/IvettexMamani/diplomado-python-2026)


,file_id,genero,zip_origen,palabras
0,pef_02121_00220669588,female,es_pe_female.zip,22
1,pef_00610_00393434719,female,es_pe_female.zip,21
2,pef_03397_01383177056,female,es_pe_female.zip,20
3,pef_09697_00427566947,female,es_pe_female.zip,21
4,pef_08421_00908604673,female,es_pe_female.zip,18
5,pef_01523_00716764604,female,es_pe_female.zip,21
6,pem_01523_00544887121,male,es_pe_male.zip,19
7,pem_08784_01328112109,male,es_pe_male.zip,20
8,pem_01208_00367733870,male,es_pe_male.zip,20
9,pem_07049_01667310146,male,es_pe_male.zip,21


### 1.3 Corpus B: inventario de audios

Hay tres fuentes. Los 2 audios `.mp4` son grabaciones propias de videos de YouTube y los 10 archivos `.ogg` / `.aac` son entrevistas breves grabadas por la estudiante, todos compartidos por WhatsApp. Los 12 `.wav` son los de OpenSLR 73. La duración se mide decodificando cada audio a 16 kHz mono, el formato que usan los modelos de voz.

In [4]:
from faster_whisper import decode_audio

SR = 16_000  # los modelos de voz esperan audio mono a 16 kHz


def cargar_audio(ruta):
    "Cualquier audio (mp4, ogg, aac, wav) -> arreglo float32 mono a 16 kHz, vía PyAV (sin instalar ffmpeg)."
    try:
        return decode_audio(str(ruta), sampling_rate=SR)
    except Exception:
        # algunos .ogg traen un último paquete dañado que PyAV rechaza; libsndfile sí los lee (como en voz.py)
        import soundfile as sf
        from scipy.signal import resample_poly
        x, sr = sf.read(str(ruta), dtype="float32")
        x = x.mean(axis=1) if x.ndim == 2 else x
        g = np.gcd(sr, SR)
        return resample_poly(x, SR // g, sr // g).astype(np.float32)


def fuente_audio(ruta):
    if ruta.suffix == ".wav":
        return "OpenSLR 73"
    return "YouTube (grabación propia)" if ruta.suffix == ".mp4" else "Entrevista propia"


FUENTES_AUDIO = ["YouTube (grabación propia)", "Entrevista propia", "OpenSLR 73"]
rutas_audio = [p for p in AUDIO_RAW.iterdir() if p.is_file()] + list(DIR_OPENSLR.glob("*.wav"))
audios = pd.DataFrame({"ruta": rutas_audio})
audios["archivo"] = audios["ruta"].map(lambda r: r.name)
audios["fuente"] = audios["ruta"].map(fuente_audio)
audios["formato"] = audios["ruta"].map(lambda r: r.suffix[1:])
audios = audios.sort_values(["fuente", "archivo"], key=lambda s: s.map(FUENTES_AUDIO.index) if s.name == "fuente" else s)
audios = audios.reset_index(drop=True)
audios["duracion_seg"] = [round(len(cargar_audio(r)) / SR, 1) for r in audios["ruta"]]

(audios.groupby("fuente", sort=False)
       .agg(archivos=("archivo", "size"), formato=("formato", lambda s: ", ".join(sorted(set(s)))),
            minutos=("duracion_seg", lambda s: round(s.sum() / 60, 1)))
       .assign(**{"% de minutos": lambda t: (100 * t["minutos"] / t["minutos"].sum()).round(0)}))

,archivos,formato,minutos,% de minutos
fuente,,,,
YouTube (grabación propia),2,mp4,27.9,54.0
Entrevista propia,10,"aac, ogg",21.6,42.0
OpenSLR 73,12,wav,1.8,4.0
